In [ ]:
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import pyarrow.compute as pc
from ecifp.utils import fetch_api_data, get_data_dir
from ecifp.fp_sim import get_bs_clusters
import matplotlib.colors as mcolors
import distinctipy
from collections import defaultdict
import molviewspec as mvs

In [ ]:
## Get the data directory
DATADIR = get_data_dir()

In [ ]:
def get_superposition_matrices(uniprot_id):
    base_url = "https://www.ebi.ac.uk/pdbe/static/superpose/matrices"
    url = f"{base_url}/{uniprot_id}"
    matrices = fetch_api_data(url)
    return matrices

## Binding sites of circadian clock oscillator protein KaiC 

In [ ]:
bs_meta_table = pq.read_table(DATADIR / "protein_bs_residues.parquet")

In [4]:
uniprot_id = 'Q79PF4'
bs_table = bs_meta_table.filter(pc.equal(bs_meta_table["protein"], uniprot_id)).select(["complex_id", "entry_id", "chem_comps", "binding_residues"]).to_pandas()

In [5]:
data_matrix = (
    pd.get_dummies(
        bs_table
        .explode('binding_residues')
        ['binding_residues']
    )
    .groupby(level=0)
    .sum()
).values

In [6]:
def get_clusters_per_entry(bs_table, clusters):
    unique_entries = bs_table['entry_id'].unique()
    entry_clusters = defaultdict(list)
    chem_comp_keys = ["auth_seq_id", "label_comp_id", "sym_op"]
    for entry in unique_entries:
        entry_meta_data = bs_table[bs_table['entry_id']==entry]
        for index, row in entry_meta_data.iterrows():
            for chem_comp in row["chem_comps"]:
                chain_id = f"{entry}_{chem_comp['auth_asym_id']}"
                entry_clusters[chain_id].append(
                    dict(
                        filter(
                            lambda item: item[0] in chem_comp_keys,
                            chem_comp.items()
                        )
                    )
                )
                entry_clusters[chain_id][-1]["cluster"] = int(clusters[index])
    
    return entry_clusters

In [7]:
superposition_matrices = get_superposition_matrices(uniprot_id)

In [8]:
def visualise_clusters(uniprot_id, entry_clusters):
    builder = mvs.create_builder()
    base_url = "https://files.wwpdb.org/download/{}.cif"

    superposition_matrices = get_superposition_matrices(uniprot_id)

    num_clusters = len(set([chem_comp['cluster'] for _, chem_comps in entry_clusters.items() for chem_comp in chem_comps]))
    cluster_colours = list(map(mcolors.to_hex, distinctipy.get_colors(num_clusters)))

    for entry_chain, chem_comps in entry_clusters.items():
        superpose_data = superposition_matrices.get(entry_chain, None)
        if superpose_data:
            matrix = np.array(superpose_data['matrix'])
            translation = matrix[:, -1][:-1].tolist()
            rotation = matrix[:, :-1][:-1].T.ravel().tolist()
            entry_id, chain_id = entry_chain.split('_')
            struct = (
                builder.download(url=base_url.format(entry_id))
                    .parse(format='mmcif')
                    .model_structure()
                    .transform(rotation=rotation,translation=translation)
            )
            chain = struct.component(
                selector=mvs.ComponentExpression(auth_asym_id=chain_id)
            )
            chain.representation(type='cartoon').opacity(opacity=0.7).color(color='grey')
            for chem_comp in chem_comps:
                ligand = struct.component(
                    selector=mvs.ComponentExpression(
                        auth_asym_id=chain_id,
                        auth_seq_id = int(chem_comp['auth_seq_id']),
                    )
                )
                (
                    ligand.representation(type="ball_and_stick",size_factor=0.5)
                        .color(color=cluster_colours[chem_comp['cluster']-1])
                )




    return builder


In [9]:
clusters = get_bs_clusters(data_matrix)

In [10]:
entry_clusters = get_clusters_per_entry(bs_table, clusters)

In [11]:
visualise_clusters(uniprot_id, entry_clusters)

<IPython.core.display.Javascript object>